# Topic: Content-Based Filtering (CBF)

## Definition (30-second explanation)
* Recommends items based on the intrinsic attributes/features of items and a user's past interaction history.
* Instead of leveraging other users' behavior, it matches the feature profile of unviewed items to items the user already liked.
* Highly standard baseline in text-heavy domains (articles, job boards, movies, books).

## Why Interviewers Ask This
* Tests understanding of the fundamental trade-offs between Collaborative Filtering and Content-Based Filtering.
* Evaluates practical feature engineering skills for unstructured data (TF-IDF, dense embeddings).
* Assesses system design awareness regarding cold start solutions and filter bubbles (overspecialization).

## Core Concepts
* **Item Profile:** Vector representation of an item’s features (e.g., TF-IDF on genres, cast, descriptions, or metadata tags).
* **User Profile:** Aggregated vector of item features the user has positively interacted with (e.g., average/weighted sum of liked item vectors).
* **Similarity Matching:** Computing distance/similarity (typically Cosine Similarity) between user profile vector and candidate item vectors.
* **Feature Weighting:** Assigning relative importance to different metadata attributes (e.g., genre weight > description weight).

## When to Use
* When item metadata is rich, descriptive, and structured/semi-structured (e.g., text, tags, genres).
* When solving the **New Item Cold Start problem** (new items have 0 user interactions but have rich metadata).
* When explainability/interpretability is critical (e.g., "Recommended because you liked movies directed by Christopher Nolan").

## Advantages
* **No Item Cold Start:** Can immediately recommend newly added items as soon as metadata is extracted.
* **No Sparsity Issues:** Does not depend on ratings from other users; works even with a single user's isolated history.
* **High Interpretability:** Recommendations are easily explainable via explicit feature overlap.

## Limitations
* **Filter Bubbles (Overspecialization):** Only recommends items similar to what the user already consumed; lacks serendipity.
* **Domain Knowledge Required:** Heavy reliance on manual feature extraction and domain-specific metadata quality.
* **New User Cold Start:** Still requires an initial profile or preference selection from a brand-new user.

## Common Comparisons
* **CBF vs. Collaborative Filtering (CF):** CF uses user-item interaction matrices (finds similar users); CBF uses item feature matrices (finds similar item attributes).
* **Item Cold Start:** CF fails completely (no interaction data); CBF handles it out-of-the-box.
* **Serendipity / Diversity:** CF introduces surprising cross-genre recommendations; CBF tends to narrow recommendations down a single path.

## Common Interview Traps
* **Trap 1:** Forgetting to combine and weight multiple feature modalities (e.g., treating short descriptions with equal weight to specific genres).
* **Trap 2:** Ignoring TF-IDF normalization before calculating dot products/cosine similarity.
* **Trap 3:** Failing to mention mitigation strategies for the "filter bubble" (e.g., adding epsilon-greedy exploration or popularity exploration).

## Python / SQL Syntax (if applicable)
```python
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

#1. Feature concatenation & TF-IDF
movies['content'] = movies['genres'] + ' ' + movies['description']
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['content'])

#2. Compute Cosine Similarity Matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

#3. Retrieve Top-N items for a target index
sim_scores = list(enumerate(cosine_sim[target_idx]))
sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:N+1]
```

## Important Formula (if applicable)
* **TF-IDF Calculation:** TF-IDF = Term Frequency (TF) * Inverse Document Frequency (IDF)
* **Cosine Similarity:** Cosine(A, B) = (A dot B) / (Magnitude of A * Magnitude of B)

## 45-Second Interview Answer
"Content-Based Filtering recommends items by matching item attributes—such as genres, tags, or text descriptions converted via TF-IDF or dense embeddings—to a user’s historical preference profile using cosine similarity. Its primary strength is completely solving the new-item cold start problem and providing clear explainability, since recommendations require zero community interaction data. However, its main drawback is overspecialization or the 'filter bubble,' where users are never recommended novel or serendipitous items outside their historical feature footprint."

## Practice Questions:

### Q1: How does content-based filtering differ from collaborative filtering?
* **Answer:**
  "Collaborative Filtering relies exclusively on user-item behavioral interaction data (e.g., clicks, purchases, ratings) to identify latent patterns across users, making it item-feature agnostic. In contrast, Content-Based Filtering relies strictly on item metadata/features and individual user preferences. CF excels at serendipity and cross-domain discovery but fails on new items (cold start), whereas CBF immediately recommends new items with rich metadata but suffers from overspecialization (filter bubbles)."
* **Common Mistakes:** Saying CBF does not suffer from *any* cold start (it still suffers from new *user* cold start).
* **Interviewer Follow-up:** "How would you combine both to get the best of both worlds in production?"
  * **Follow-up Answer:** "I would use a two-stage hybrid funnel. I'd use Content-Based Filtering as the initial candidate generator to handle cold-start items and ensure relevance, and then pass those candidates to a Collaborative Filtering or Deep Learning ranking model to personalize the final order based on community behavior."

### Q2: What is the filter bubble problem and how would you solve it?
* **Answer:**
  "The filter bubble (or overspecialization) occurs when a CBF system continuously recommends items identical in feature space to what the user has already consumed, preventing them from discovering new genres or categories. To solve it, I would:
  1. Inject an epsilon-greedy exploration policy (e.g., 10% of recommendations drawn from trending/popular items outside their profile).
  2. Implement a diversity re-ranking penalty (e.g., Maximal Marginal Relevance - MMR) to penalize redundant items."
* **Common Mistakes:** Only identifying the problem without concrete algorithmic solutions like MMR or exploration policies.
* **Interviewer Follow-up:** "How does Maximal Marginal Relevance (MMR) mathematically balance relevance vs diversity?"
  * **Follow-up Answer:** "MMR evaluates items by applying a penalty term. It calculates the similarity of a candidate item to the user's profile to maximize relevance, but explicitly subtracts the maximum similarity between that candidate and the items *already added* to the recommendation list to enforce diversity."

### Q3: How do you build a user profile in content-based filtering?
* **Answer:**
  "A user profile is constructed in the same vector space as the items. For explicit ratings, it is typically computed as the weighted average of the feature vectors of items the user has interacted with, where the weights correspond to their ratings or engagement strength. For implicit feedback, decay factors can be applied over time so recent interactions carry higher weight."
* **Common Mistakes:** Forgetting that the user profile and item profile must share the exact same vector dimension to compute similarity.
* **Interviewer Follow-up:** "How do you handle negative feedback (e.g., thumbs down) when constructing the vector?"
  * **Follow-up Answer:** "I would apply negative weights (e.g., -1 or -0.5) to the item's feature vector before aggregating, which mathematically pushes the user's profile vector away from the features associated with the disliked item."

### Q4: What features would you use to build a content-based movie recommender?
* **Answer:**
  "I would engineer features across three main categories:
  1. Categorical Metadata: Genres, directors, top cast members (encoded via one-hot encoding).
  2. Unstructured Text: Plot summaries, reviews (encoded via TF-IDF or transformer-based sentence embeddings).
  3. Numerical Features: Release year, average critic score (normalized using min-max scaling).
  Finally, I would weight higher-signal features (like Director/Genre) more heavily than broad text descriptions before concatenating."
* **Common Mistakes:** Treating all features as raw text and dumping them into a single unweighted TF-IDF vectorizer.
* **Interviewer Follow-up:** "Why might dense transformer embeddings outperform TF-IDF for plot descriptions?"
  * **Follow-up Answer:** "TF-IDF relies on exact keyword matching and ignores word order and semantics. Dense embeddings (like BERT) capture semantic meaning, so phrases like 'funny movie' and 'hilarious film' map to similar vectors even if the exact words differ."

### Q5: When would you prefer content-based over collaborative filtering?
* **Answer:**
  "I prefer Content-Based Filtering in specific scenarios:
  1. New/Frequent Catalog Turnover: In domains like news or job postings where items are constantly newly created and have zero initial interactions.
  2. Niche/Sparse User Base: When the platform has very low user overlap, making collaborative co-occurrence matrices too sparse to factorize.
  3. Strict Explainability Requirements: When the system must provide explicit, deterministic reasons for every recommendation."
* **Common Mistakes:** Forgetting domain examples like news or job boards where item turnover is too fast for CF.
* **Interviewer Follow-up:** "In a news recommender, how do you handle rapidly decaying content value?"
  * **Follow-up Answer:** "I would introduce a time-decay factor (like an exponential decay function) to the final similarity score, ensuring older articles are heavily penalized regardless of how well their features match the user profile."